# Experiment 008 — KL-Selective JRR

Tests whether only the KL-increasing component of the nonlinear JRR remainder should be removed. The original JRR held-out is not reused as confirmatory data; a fresh held-out split is frozen in the repository.

In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
print('cwd:', os.getcwd())

## 0. Restore the frozen sentiment direction if needed

In [ ]:
from pathlib import Path
direction = Path('results/sentiment_direction.pt')
if not direction.exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using frozen direction:', direction)

## 1. Unit tests

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_selective_jrr.py','tests/test_jrr.py'], check=True)

## 2. Real-model gradient preflight

Checks that the KL gradient agrees with an independent finite-difference directional derivative and that the selected correction is orthogonal to transported `Jv`.

In [ ]:
subprocess.run([sys.executable,'scripts/preflight_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml'], check=True)

## 3. Calibration

Compares additive, full JRR, and KL-JRR on the existing calibration prompts only. No beta or layer sweep is performed.

In [ ]:
subprocess.run([sys.executable,'scripts/run_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml','--phase','calibration'], check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
cal = json.loads(Path('results/selective_jrr/calibration_summary.json').read_text())
print(json.dumps(cal, indent=2))
display(pd.read_csv('results/selective_jrr/calibration_same_alpha.csv'))
display(pd.read_csv('results/selective_jrr/calibration_aggregate.csv'))
display(Image(filename='results/selective_jrr/calibration_pareto.png'))

## 4. Fresh held-out evaluation

This uses 12 prompts and seeds 101/211 that were frozen only after Experiment 007 was analyzed. The cell does nothing if the calibration gate failed.

In [ ]:
if cal['go_to_new_heldout']:
    subprocess.run([sys.executable,'scripts/run_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml','--phase','evaluation'], check=True)
    print('Fresh held-out complete.')
else:
    print('STOP: calibration did not support opening the new held-out. Do not use --force for the reported result.')

In [ ]:
if cal['go_to_new_heldout']:
    ev = json.loads(Path('results/selective_jrr/evaluation_summary.json').read_text())
    print(json.dumps(ev, indent=2))
    display(pd.read_csv('results/selective_jrr/evaluation_same_alpha.csv'))
    display(pd.read_csv('results/selective_jrr/evaluation_aggregate.csv'))
    display(Image(filename='results/selective_jrr/evaluation_pareto.png'))

## 5. Package results

Send the ZIP back for analysis even if calibration fails.

In [ ]:
import shutil
archive = shutil.make_archive('/content/selective_jrr_results','zip','results/selective_jrr')
print('Created:', archive)